# Notebook 05: Modelling Data Preparation

## Objective

This notebook prepares separate datasets for the econometric and machine learning stages of the research.

The econometric dataset will support the analysis of symmetric and asymmetric exchange-rate pass-through. The machine learning dataset will support time-aware model training and evaluation.

This notebook focuses on modelling decisions and does not repeat the exploratory analysis completed in Notebook 04.

In [1]:
# Import Libraries
from pathlib import Path
import pandas as pd
import numpy as np

The feature-engineered dataset produced in Notebook 03 is used as the input for modelling preparation. Notebook 04 is not used as a data source because it contains exploratory transformations created for analysis rather than for the final modelling pipeline.

In [2]:
# create file path
DATA_DIR = Path("../data/processed")
featured_df = DATA_DIR / "featured_data.csv"

In [3]:
data = pd.read_csv(
    featured_df, 
    parse_dates=["Date"]
    )

data = data.sort_values(
    ["Eight digit code", "Date"]
).reset_index(drop=True)

In [4]:
# preview the dataset
data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate,Year,...,ExchangeRate_Lag1,ExchangeRate_Lag3,ExchangeRate_Lag6,CPI_Lag1,CPI_Lag3,ExchangeRate_Change,Depreciation,Appreciation,ExchangeRate_MA3,ExchangeRate_STD3
0,2015-07-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.2,12.452687,2015,...,12.303048,12.012553,11.566500,57.0,57.0,0.012163,0.012163,0.000000,12.241892,0.247116
1,2015-08-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,59.3,12.914595,2015,...,12.452687,11.969940,11.577440,57.2,57.3,0.037093,0.037093,0.000000,12.556777,0.318784
2,2015-09-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,58.9,13.610852,2015,...,12.914595,12.303048,12.068727,59.3,57.0,0.053912,0.053912,0.000000,12.992711,0.583021
3,2015-10-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,59.5,13.504323,2015,...,13.610852,12.452687,12.012553,58.9,57.2,-0.007827,0.000000,0.007827,13.343257,0.375034
4,2015-11-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,58.9,14.125733,2015,...,13.504323,12.914595,11.969940,59.5,59.3,0.046016,0.046016,0.000000,13.746969,0.332316


In [5]:
# confirm the dataset shape
print("Dataset shape:", data.shape)

Dataset shape: (14150, 22)


Each observation should represent one eight-digit food item in one month.

The eight-digit code is used as the item identifier because multiple food items may share the same subclass description. Before preparing either modelling dataset, the combination of eight-digit code and date must therefore be unique.

In [6]:
# check the item-month structure
modelling_key = ["Eight digit code", "Date"]

duplicate_count = data.duplicated(subset=modelling_key).sum()

print(
    "Unique eight-digit codes:",
    data["Eight digit code"].nunique()
)

print(
    "Unique subclass descriptions:",
    data["SubclassDescription"].nunique()
)

print(
    "Duplicate item-month observations:",
    duplicate_count
)

Unique eight-digit codes: 134
Unique subclass descriptions: 49
Duplicate item-month observations: 0


### Interpretation of the Modelling Unit

The dataset contains 134 unique eight-digit food item codes across 49 subclass descriptions. This confirms that multiple individual food items may belong to the same subclass.

The combination of `Eight digit code` and `Date` is therefore used as the modelling key. This preserves the individual CPI series within each food subclass.

No duplicate item-month observations were identified, confirming that each row represents a unique food item observed in a specific month.

In [7]:
# Summarize the coverage of each food item
item_coverage = (
    data.groupby("Eight digit code")
    .agg(
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        observations=("Date", "count")
    )
    .reset_index()
)

year_difference = (
    item_coverage["end_date"].dt.year - item_coverage["start_date"].dt.year
)

month_difference = (
    item_coverage["end_date"].dt.month - item_coverage["start_date"].dt.month
)

item_coverage["expected_months"] = (year_difference * 12 + month_difference + 1 )

item_coverage["missing_months"] = (
    item_coverage["expected_months"] - item_coverage["observations"]
)

In [8]:
# Review the panel structure
print("Overall start date:", data["Date"].min().date())
print("Overall end date:", data["Date"].max().date())
print("Unique months:", data["Date"].nunique())

print(
    "Items with missing months:",
    (item_coverage["missing_months"] > 0).sum()
)

item_coverage["observations"].describe()

Overall start date: 2015-07-01
Overall end date: 2025-12-01
Unique months: 126
Items with missing months: 0


count    134.000000
mean     105.597015
std       40.044359
min       10.000000
25%      105.000000
50%      126.000000
75%      126.000000
max      126.000000
Name: observations, dtype: float64

In [9]:
# view items with the least observations
item_coverage.sort_values(
    ["observations", "Eight digit code"]
).head(10)

,Eight digit code,start_date,end_date,observations,expected_months,missing_months
1,1111202,2025-03-01,2025-12-01,10,10,0
4,1112301,2025-03-01,2025-12-01,10,10,0
32,1124002,2025-03-01,2025-12-01,10,10,0
33,1124003,2025-03-01,2025-12-01,10,10,0
38,1125104,2025-03-01,2025-12-01,10,10,0
39,1125105,2025-03-01,2025-12-01,10,10,0
41,1125901,2025-03-01,2025-12-01,10,10,0
42,1125902,2025-03-01,2025-12-01,10,10,0
47,1133902,2025-03-01,2025-12-01,10,10,0
48,1134101,2025-03-01,2025-12-01,10,10,0


### Interpretation of the Panel Structure

The dataset covers 126 months from July 2015 to December 2025 and contains no internal gaps within the individual food-item series.

However, the number of observations per item ranges from 10 to 126. The panel is therefore unbalanced because some food items enter or leave the dataset at different points during the study period.

The shorter series should not be treated as missing data because their observations are continuous within their available periods. No additional months will be imputed.

In [10]:
# summarise item coverage patterns
coverage_patterns = (
    item_coverage.groupby(
        ["start_date", "end_date", "observations"]
    )
    .size()
    .reset_index(name="item_count")
    .sort_values(
        ["start_date", "end_date", "observations"]
    )
)

coverage_patterns

,start_date,end_date,observations,item_count
0,2015-07-01,2025-12-01,126,95
1,2017-04-01,2025-12-01,105,17
2,2022-04-01,2025-12-01,45,5
3,2025-03-01,2025-12-01,10,17


In [11]:
# Count items covering the full modelling period
overall_start_date = data["Date"].min()
overall_end_date = data["Date"].max()

full_period_items = (
    item_coverage["start_date"].eq(overall_start_date)
    & item_coverage["end_date"].eq(overall_end_date)
).sum()

print("Items covering the full period:", full_period_items)
print(
    "Items with fewer than 60 observations:",
    (item_coverage["observations"] < 60).sum()
)
print(
    "Items introduced during 2025:",
    (item_coverage["start_date"].dt.year == 2025).sum()
)

Items covering the full period: 95
Items with fewer than 60 observations: 22
Items introduced during 2025: 17


### Interpretation of Item Coverage

The panel contains four groups of food items based on their starting dates. Ninety-five items cover the full 126 month modelling period, while the remaining items entered the dataset in 2017, 2022 or 2025.

All item series continue until December 2025 and contain no internal missing months. The unequal series lengths therefore reflect the later introduction of certain food items rather than incomplete observations within their available periods.

The 2022 and 2025 series may be too short for reliable item-level econometric modelling. Their food-category coverage must be examined before deciding whether they should be excluded.

In [12]:
# Add food-category labels to the coverage summary
item_labels = data[
    [
        "Eight digit code",
        "ClassDescription",
        "SubclassDescription"
    ]
].drop_duplicates()

item_details = item_coverage.merge(
    item_labels,
    on="Eight digit code",
    how="left",
    validate="one_to_one"
)

In [13]:
# compare category coverage by series length
coverage_by_length = (
    item_details.groupby(
        ["start_date", "observations"]
    )
    .agg(
        item_count=("Eight digit code", "nunique"),
        subclass_count=("SubclassDescription", "nunique")
    )
    .reset_index()
    .sort_values("start_date")
)

coverage_by_length

,start_date,observations,item_count,subclass_count
0,2015-07-01,126,95,44
1,2017-04-01,105,17,10
2,2022-04-01,45,5,4
3,2025-03-01,10,17,13


In [14]:
# view items with fewer than 100 observations
short_series = (
    item_details.loc[
        item_details["observations"] < 100,
        [
            "Eight digit code",
            "ClassDescription",
            "SubclassDescription",
            "start_date",
            "observations"
        ]
    ]
    .sort_values(
        ["start_date", "SubclassDescription"]
    )
)

short_series

,Eight digit code,ClassDescription,SubclassDescription,start_date,observations
110,1192301,Other food,Baby food,2022-04-01,45
127,1220102,Coffee,Coffee and coffee substitutes,2022-04-01,45
5,1112601,Cereal products,Flour of cereals,2022-04-01,45
6,1112602,Cereal products,Flour of cereals,2022-04-01,45
98,1183901,"Sugar, confectionery and desserts","Jams, fruit jellies, marmelades, fruit puree a...",2022-04-01,45
68,1152101,Oils and fats,Butter and other fats and oils derived from milk,2025-03-01,10
1,1111202,Cereal products,Cereals,2025-03-01,10
59,1145005,"Milk, other dairy products and eggs",Cheese,2025-03-01,10
47,1133902,Fish and other seafood,Fish preparations,2025-03-01,10
4,1112301,Cereal products,Flour of cereals,2025-03-01,10


### Interpretation of Category Coverage

The 95 full-period items represent 44 food subclasses. The items beginning in 2017 provide a further 17 item-level series across 10 subclasses.

The shorter series beginning in 2022 and 2025 include items from several important food categories. However, some of these subclasses are already represented by longer item-level series.

Before excluding short series from econometric modelling, the combined subclass coverage of items with at least 100 observations must be confirmed.

In [15]:
# separate longer and shorter item series
long_series_items = item_details[
    item_details["observations"] >= 100
].copy()

short_series_items = item_details[
    item_details["observations"] < 100
].copy()

all_subclasses = set(
    item_details["SubclassDescription"].unique()
)

long_series_subclasses = set(
    long_series_items["SubclassDescription"].unique()
)

subclasses_without_long_series = sorted(
    all_subclasses - long_series_subclasses
)

In [16]:
# check subclass coverage in the longer series
print("Long-series items:", len(long_series_items))
print(
    "Subclasses represented:",
    long_series_items["SubclassDescription"].nunique()
)
print(
    "Subclasses without a long series:",
    len(subclasses_without_long_series)
)

subclasses_without_long_series

Long-series items: 112
Subclasses represented: 46
Subclasses without a long series: 3


['Butter and other fats and oils derived from milk',
 'Jams, fruit jellies, marmelades, fruit puree and paste',
 'Other seafood']

### Interpretation of Subclass Retention

The longer item series represent 46 of the 49 food subclasses. Three subclasses are represented only by series containing 45 or fewer monthly observations.

Butter and other fats and oils derived from milk and other seafood contain only 10 observations. Jams, fruit jellies, marmalades, fruit purée and paste contains item series with 45 and 10 observations.

These series are retained in the exploratory dataset but excluded from the candidate econometric sample because they do not provide sufficient time coverage for reliable category-level ARDL or NARDL estimation.

The main econometric and machine learning comparison will use the common set of 46 eligible subclasses. Shorter series may be considered separately during the extended machine learning analysis.

In [17]:
# select items with at least 100 observations
eligible_econometric_codes = long_series_items[
    "Eight digit code"
].tolist()

econometric_candidate_data = data[
    data["Eight digit code"].isin(
        eligible_econometric_codes
    )
].copy()

In [18]:
# confirm the econometric candidate sample
econometric_item_counts = (
    econometric_candidate_data.groupby(
        "Eight digit code"
    )
    .size()
)

print(
    "Observations:",
    f"{len(econometric_candidate_data):,}"
)
print(
    "Eight-digit items:",
    econometric_candidate_data[
        "Eight digit code"
    ].nunique()
)
print(
    "Food subclasses:",
    econometric_candidate_data[
        "SubclassDescription"
    ].nunique()
)
print(
    "Minimum observations per item:",
    econometric_item_counts.min()
)
print(
    "Maximum observations per item:",
    econometric_item_counts.max()
)

Observations: 13,755
Eight-digit items: 112
Food subclasses: 46
Minimum observations per item: 105
Maximum observations per item: 126


### Interpretation of the Econometric Candidate Sample

The econometric candidate sample contains 13,755 observations from 112 food items across 46 subclasses.

Each item provides between 105 and 126 monthly observations. Although all series are sufficiently long for further econometric assessment, their starting dates differ.

Using different estimation periods could make category-level results less comparable because the models would be exposed to different exchange-rate conditions. A common estimation period is therefore prepared for the primary econometric analysis.

In [19]:
# create the common-period econometric sample
common_start_date = long_series_items[
    "start_date"
].max()

common_end_date = econometric_candidate_data[
    "Date"
].max()

econometric_common_data = econometric_candidate_data[
    econometric_candidate_data["Date"].between(
        common_start_date,
        common_end_date
    )
].copy()

In [20]:
# create the full-period robustness sample
full_period_codes = item_details.loc[
    item_details["start_date"].eq(data["Date"].min()),
    "Eight digit code"
]

econometric_full_period_data = data[
    data["Eight digit code"].isin(
        full_period_codes
    )
].copy()

In [21]:
# Compare the econometric samples
print("Primary common-period sample")
print("Observations:", f"{len(econometric_common_data):,}")
print(
    "Items:",
    econometric_common_data["Eight digit code"].nunique()
)
print(
    "Subclasses:",
    econometric_common_data["SubclassDescription"].nunique()
)
print(
    "Period:",
    econometric_common_data["Date"].min().date(),
    "to",
    econometric_common_data["Date"].max().date()
)

print("\nFull-period robustness sample")
print("Observations:", f"{len(econometric_full_period_data):,}")
print(
    "Items:",
    econometric_full_period_data["Eight digit code"].nunique()
)
print(
    "Subclasses:",
    econometric_full_period_data["SubclassDescription"].nunique()
)
print(
    "Period:",
    econometric_full_period_data["Date"].min().date(),
    "to",
    econometric_full_period_data["Date"].max().date()
)

Primary common-period sample
Observations: 11,760
Items: 112
Subclasses: 46
Period: 2017-04-01 to 2025-12-01

Full-period robustness sample
Observations: 11,970
Items: 95
Subclasses: 44
Period: 2015-07-01 to 2025-12-01


### Interpretation of the Econometric Samples

The primary econometric sample contains 11,760 observations from 112 food items across 46 subclasses. Every item contributes the same 105 months from April 2017 to December 2025.

The full-period robustness sample contains 11,970 observations from 95 food items across 44 subclasses. Every item contributes 126 months from July 2015 to December 2025.

The common-period sample will support the main category comparison, while the full-period sample will be used to assess whether including the earlier observations changes the econometric findings.

Two related dependent variables are required.

`Log_CPI` represents the food-price level and will be used when estimating long-run ARDL and NARDL relationships.

`Food_Inflation_Pct` represents the monthly log change in CPI. It will be used for short-run analysis and as the machine learning prediction target.

Using monthly log changes makes price movements more comparable across food items with different CPI levels.

The monthly food-inflation measure is calculated as:

Food Inflation = 100 × [log(CPI at time t) − log(CPI at time t−1)]

In [22]:
# check values before applying logarithms
non_positive_cpi = (
    data[["CPI", "CPI_Lag1"]] <= 0
).any(axis=1).sum()

print("Non-positive CPI observations:", non_positive_cpi)

Non-positive CPI observations: 0


In [23]:
# create the dependent variables
data["Log_CPI"] = np.log(data["CPI"])
data["Log_CPI_Lag1"] = np.log(data["CPI_Lag1"])

data["Food_Inflation_Pct"] = 100 * (
    data["Log_CPI"]
    - data["Log_CPI_Lag1"]
)

In [24]:
# inspect the dependent variables
target_columns = [
    "Date",
    "Eight digit code",
    "CPI",
    "CPI_Lag1",
    "Log_CPI",
    "Food_Inflation_Pct"
]

display(data[target_columns].head())

print(
    "\nMissing food-inflation values:",
    data["Food_Inflation_Pct"].isna().sum()
)

data[
    ["Log_CPI", "Food_Inflation_Pct"]
].describe().T

,Date,Eight digit code,CPI,CPI_Lag1,Log_CPI,Food_Inflation_Pct
0,2015-07-01,1111201,57.2,57.0,4.046554,0.350263
1,2015-08-01,1111201,59.3,57.2,4.082609,3.605541
2,2015-09-01,1111201,58.9,59.3,4.075841,-0.676822
3,2015-10-01,1111201,59.5,58.9,4.085976,1.013522
4,2015-11-01,1111201,58.9,59.5,4.075841,-1.013522



Missing food-inflation values: 0


,count,mean,std,min,25%,50%,75%,max
Log_CPI,14150.0,4.368898,0.199301,3.691376,4.220977,4.365643,4.55703,5.148076
Food_Inflation_Pct,14150.0,0.475409,2.431204,-23.142962,-0.408894,0.370600,1.31755,28.175545


### Interpretation of the Dependent Variables

The log CPI and monthly food-inflation variables were calculated successfully, with no missing observations.

Average monthly food inflation is approximately 0.48%, while the median is approximately 0.37%. The positive average indicates that food prices generally increased over the modelling period.

Monthly food-inflation values range from approximately -23.14% to 28.18%, showing that some food items experienced substantial monthly price changes.

These observations are retained because large food-price movements may represent genuine market behaviour. Their effect on model performance will be assessed during the econometric diagnostics and machine learning evaluation.

In [25]:
# check exchange-rate consistency within each month
exchange_rate_consistency = (
    data.groupby("Date")[
        ["ExchangeRate", "ExchangeRate_Lag1"]
    ]
    .nunique()
)

inconsistent_exchange_months = (
    exchange_rate_consistency > 1
).any(axis=1).sum()

print(
    "Months with inconsistent exchange rates:",
    inconsistent_exchange_months
)

Months with inconsistent exchange rates: 0


In [26]:
# create the monthly exchange-rate series
monthly_exchange_rate = (
    data[
        [
            "Date",
            "ExchangeRate",
            "ExchangeRate_Lag1"
        ]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Monthly exchange-rate observations:",
    len(monthly_exchange_rate)
)

Monthly exchange-rate observations: 126


In [27]:
# create exchange-rate transformations
monthly_exchange_rate["Log_ExchangeRate"] = np.log(
    monthly_exchange_rate["ExchangeRate"]
)

monthly_exchange_rate["ExchangeRate_Log_Change_Pct"] = 100 * (
    np.log(monthly_exchange_rate["ExchangeRate"])
    - np.log(monthly_exchange_rate["ExchangeRate_Lag1"])
)

monthly_exchange_rate["Depreciation_Shock_Pct"] = (
    monthly_exchange_rate["ExchangeRate_Log_Change_Pct"]
    .clip(lower=0)
)

monthly_exchange_rate["Appreciation_Shock_Pct"] = (
    monthly_exchange_rate["ExchangeRate_Log_Change_Pct"]
    .clip(upper=0)
)

monthly_exchange_rate["Appreciation_Magnitude_Pct"] = (
    monthly_exchange_rate["Appreciation_Shock_Pct"].abs()
)

In [28]:
# validate the asymmetric decomposition
decomposition_error = (
    monthly_exchange_rate["ExchangeRate_Log_Change_Pct"]
    - (
        monthly_exchange_rate["Depreciation_Shock_Pct"]
        + monthly_exchange_rate["Appreciation_Shock_Pct"]
    )
).abs().max()

print(
    "Depreciation months:",
    (
        monthly_exchange_rate["Depreciation_Shock_Pct"] > 0
    ).sum()
)

print(
    "Appreciation months:",
    (
        monthly_exchange_rate["Appreciation_Shock_Pct"] < 0
    ).sum()
)

print(
    "No-change months:",
    (
        monthly_exchange_rate["ExchangeRate_Log_Change_Pct"] == 0
    ).sum()
)

print(
    "Maximum decomposition error:",
    decomposition_error
)

monthly_exchange_rate.head()

Depreciation months: 55
Appreciation months: 71
No-change months: 0
Maximum decomposition error: 0.0


,Date,ExchangeRate,ExchangeRate_Lag1,Log_ExchangeRate,ExchangeRate_Log_Change_Pct,Depreciation_Shock_Pct,Appreciation_Shock_Pct,Appreciation_Magnitude_Pct
0,2015-07-01,12.452687,12.303048,2.521936,1.208941,1.208941,0.000000,0.000000
1,2015-08-01,12.914595,12.452687,2.558358,3.642165,3.642165,0.000000,0.000000
2,2015-09-01,13.610852,12.914595,2.610867,5.250938,5.250938,0.000000,0.000000
3,2015-10-01,13.504323,13.610852,2.603010,-0.785761,0.000000,-0.785761,0.785761
4,2015-11-01,14.125733,13.504323,2.647998,4.498836,4.498836,0.000000,0.000000


### Interpretation of the Exchange-Rate Variables

The 14,150 item-level observations correspond to 126 unique monthly exchange-rate observations. No conflicting exchange-rate values were identified within any month.

The Rand depreciated against the US dollar in 55 months and appreciated in 71 months. There were no months in which the monthly average exchange rate remained exactly unchanged.

The signed exchange-rate change was successfully separated into depreciation and appreciation shocks, with a maximum decomposition error of zero.

The frequency of depreciation and appreciation does not indicate their relative effect on food prices. Their accumulated magnitudes and subsequent pass-through must be assessed through the econometric and machine learning models.

In [29]:
# create the cumulative NARDL components
monthly_exchange_rate[
    "ExchangeRate_Positive_Cumulative_Pct"
] = (
    monthly_exchange_rate["Depreciation_Shock_Pct"]
    .cumsum()
)

monthly_exchange_rate[
    "ExchangeRate_Negative_Cumulative_Pct"
] = (
    monthly_exchange_rate["Appreciation_Shock_Pct"]
    .cumsum()
)

In [30]:
# validate the cumulative decomposition
initial_log_exchange_rate = np.log(
    monthly_exchange_rate.loc[
        0,
        "ExchangeRate_Lag1"
    ]
)

direct_cumulative_change = 100 * (
    monthly_exchange_rate["Log_ExchangeRate"]
    - initial_log_exchange_rate
)

reconstructed_cumulative_change = (
    monthly_exchange_rate[
        "ExchangeRate_Positive_Cumulative_Pct"
    ]
    + monthly_exchange_rate[
        "ExchangeRate_Negative_Cumulative_Pct"
    ]
)

cumulative_decomposition_error = (
    direct_cumulative_change
    - reconstructed_cumulative_change
).abs().max()

print(
    "Cumulative depreciation:",
    monthly_exchange_rate[
        "ExchangeRate_Positive_Cumulative_Pct"
    ].iloc[-1]
)

print(
    "Cumulative appreciation:",
    monthly_exchange_rate[
        "ExchangeRate_Negative_Cumulative_Pct"
    ].iloc[-1]
)

print(
    "Net log exchange-rate change:",
    reconstructed_cumulative_change.iloc[-1]
)

print(
    "Maximum cumulative decomposition error:",
    cumulative_decomposition_error
)

Cumulative depreciation: 183.204582398047
Cumulative appreciation: -151.72060274986362
Net log exchange-rate change: 31.48397964818338
Maximum cumulative decomposition error: 7.105427357601002e-14


In [31]:
# preview the cumulative components
monthly_exchange_rate[
    [
        "Date",
        "ExchangeRate_Log_Change_Pct",
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct"
    ]
].tail()

,Date,ExchangeRate_Log_Change_Pct,ExchangeRate_Positive_Cumulative_Pct,ExchangeRate_Negative_Cumulative_Pct
121,2025-08-01,-0.158029,183.204582,-146.656649
122,2025-09-01,-1.522017,183.204582,-148.178666
123,2025-10-01,-1.025553,183.204582,-149.204219
124,2025-11-01,-0.318583,183.204582,-149.522801
125,2025-12-01,-2.197802,183.204582,-151.720603


### Interpretation of the NARDL Components

Cumulative Rand depreciation amounted to approximately 183.20 log-percentage points, while cumulative appreciation amounted to approximately -151.72 log-percentage points.

Together, these components produce a net exchange-rate log change of approximately 31.48 percentage points over the available period.

The cumulative decomposition error is effectively zero. The small reported value results from floating-point precision rather than an error in the calculation.

The positive and negative components will allow the NARDL models to estimate whether food prices respond differently to Rand depreciation and appreciation.

In [32]:
# select exchange-rate modelling variables
exchange_rate_model_columns = [
    "Date",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
    "Appreciation_Magnitude_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

model_data = data.merge(
    monthly_exchange_rate[
        exchange_rate_model_columns
    ],
    on="Date",
    how="left",
    validate="many_to_one"
)

In [33]:
# validate the merged modelling data
merged_variable_names = exchange_rate_model_columns[1:]

missing_merged_values = model_data[
    merged_variable_names
].isna().sum()

duplicate_model_rows = model_data.duplicated(
    subset=["Eight digit code", "Date"]
).sum()

print("Rows before merge:", len(data))
print("Rows after merge:", len(model_data))
print(
    "Duplicate item-month observations:",
    duplicate_model_rows
)

missing_merged_values

Rows before merge: 14150
Rows after merge: 14150
Duplicate item-month observations: 0


Log_ExchangeRate                        0
ExchangeRate_Log_Change_Pct             0
Depreciation_Shock_Pct                  0
Appreciation_Shock_Pct                  0
Appreciation_Magnitude_Pct              0
ExchangeRate_Positive_Cumulative_Pct    0
ExchangeRate_Negative_Cumulative_Pct    0
dtype: int64

In [34]:
# preview the merged modelling variables
model_data[
    [
        "Date",
        "Eight digit code",
        "Log_CPI",
        "Food_Inflation_Pct",
        "Log_ExchangeRate",
        "ExchangeRate_Log_Change_Pct",
        "Depreciation_Shock_Pct",
        "Appreciation_Shock_Pct",
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct"
    ]
].head()

,Date,Eight digit code,Log_CPI,Food_Inflation_Pct,Log_ExchangeRate,ExchangeRate_Log_Change_Pct,Depreciation_Shock_Pct,Appreciation_Shock_Pct,ExchangeRate_Positive_Cumulative_Pct,ExchangeRate_Negative_Cumulative_Pct
0,2015-07-01,1111201,4.046554,0.350263,2.521936,1.208941,1.208941,0.000000,1.208941,0.000000
1,2015-08-01,1111201,4.082609,3.605541,2.558358,3.642165,3.642165,0.000000,4.851106,0.000000
2,2015-09-01,1111201,4.075841,-0.676822,2.610867,5.250938,5.250938,0.000000,10.102044,0.000000
3,2015-10-01,1111201,4.085976,1.013522,2.603010,-0.785761,0.000000,-0.785761,10.102044,-0.785761
4,2015-11-01,1111201,4.075841,-1.013522,2.647998,4.498836,4.498836,0.000000,14.600880,-0.785761


### Interpretation of the Modelling-Variable Merge

The merge retained all 14,150 observations and introduced no duplicate item-month records.

All dependent-variable and exchange-rate fields were matched successfully, with no missing values in the newly merged columns.

The resulting dataframe contains the variables required to prepare the separate econometric and machine learning datasets.

In [35]:
# select eligible item-level observations
econometric_item_data = model_data[
    model_data["Eight digit code"].isin(
        eligible_econometric_codes
    )
    & model_data["Date"].between(
        common_start_date,
        common_end_date
    )
].copy()

print(
    "Eligible item-level observations:",
    f"{len(econometric_item_data):,}"
)

Eligible item-level observations: 11,760


In [36]:
# check item labels and weights
weight_counts = (
    econometric_item_data.groupby(
        "Eight digit code"
    )["Weight"]
    .nunique(dropna=False)
)

subclass_counts = (
    econometric_item_data.groupby(
        "Eight digit code"
    )["SubclassDescription"]
    .nunique(dropna=False)
)

print(
    "Items with changing weights:",
    (weight_counts > 1).sum()
)
print(
    "Items assigned to multiple subclasses:",
    (subclass_counts > 1).sum()
)
print(
    "Missing weight observations:",
    econometric_item_data["Weight"].isna().sum()
)
print(
    "Non-positive weight observations:",
    (econometric_item_data["Weight"] <= 0).sum()
)

Items with changing weights: 0
Items assigned to multiple subclasses: 0
Missing weight observations: 0
Non-positive weight observations: 0


In [37]:
# summarise the item structure within subclasses
item_reference = (
    econometric_item_data[
        [
            "Eight digit code",
            "ClassDescription",
            "SubclassDescription",
            "Weight"
        ]
    ]
    .drop_duplicates()
)

subclass_structure = (
    item_reference.groupby(
        "SubclassDescription"
    )
    .agg(
        item_count=("Eight digit code", "nunique"),
        subclass_weight=("Weight", "sum")
    )
    .sort_values("item_count")
)

print("Food subclasses:", len(subclass_structure))
print(
    "Subclasses represented by one item:",
    (subclass_structure["item_count"] == 1).sum()
)
print(
    "Maximum items in one subclass:",
    subclass_structure["item_count"].max()
)

subclass_structure.head(10)

Food subclasses: 46
Subclasses represented by one item: 14
Maximum items in one subclass: 9


,item_count,subclass_weight
SubclassDescription,,
Cereals,1,0.540243
"Dates, figs and tropical fruits, fresh",1,0.173193
Coffee and coffee substitutes,1,0.225329
Eggs,1,0.490775
Fish preparations,1,0.017047
"Ice, ice cream and sorbet",1,0.074819
Other non-alcoholic beverages,1,0.059675
"Other fruits, fresh",1,0.269220
"Nut puree, nut butter and nut pastes",1,0.079284


### Interpretation of Item Weights

The econometric sample contains 11,760 eligible item-level observations.

No item-level weights change during the common modelling period, and each item remains assigned to one food subclass. All weights are present and positive.

Fourteen subclasses are represented by one item, while the remaining subclasses contain between two and nine items. Weighted aggregation is therefore required to ensure that items with greater expenditure importance contribute more to their subclass CPI.